# Sales30 seated-v3
Upload `vsign_sales30_seated_v3.zip` thành **Kaggle Dataset riêng tư**, attach vào notebook này và bật GPU. Internet chỉ cần cho cài thư viện; không cần tải pretrained weights.

3.520 clip: train 2.425, validation 744, test đứng 351. HA/Nhi train, Như validation. Nhi thiếu từ 27 Cung cấp và 28 Thay đổi. Khởi tạo từ seated-v2 best epoch 5; optimizer và scheduler mới.

Kết quả là development-only. Như đã được dùng để chọn seated-v2 nên không phải người hoàn toàn chưa thấy. Tập test đứng không dùng để chọn epoch hoặc tham số; không đổi runtime trước khi đánh giá camera A/B cố định.


In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

INPUT_ROOT = Path(os.environ.get('VSIGN_INPUT_ROOT', '/kaggle/input'))
WORK_ROOT = Path(os.environ.get('VSIGN_WORKING_ROOT', '/kaggle/working'))
ROOT = WORK_ROOT / 'VSign-seated-v3'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
unpacked = []
for path in INPUT_ROOT.rglob('bundle_inventory.json'):
    payload = json.loads(path.read_text(encoding='utf-8'))
    if payload.get('dataset_version') == 'seated-v3':
        unpacked.append(path.parent)
archives = list(INPUT_ROOT.rglob('vsign_sales30_seated_v3.zip'))
helper_name = 'ml/scripts/setup_sales30_seated_v3_kaggle.py'
if len(unpacked) == 1:
    SOURCE = unpacked[0]
    inventory = json.loads((SOURCE / 'bundle_inventory.json').read_text(encoding='utf-8'))
    helper_bytes = (SOURCE / helper_name).read_bytes()
elif not unpacked and len(archives) == 1:
    SOURCE = archives[0]
    with zipfile.ZipFile(SOURCE) as archive:
        inventory = json.loads(archive.read('VSign/bundle_inventory.json'))
        helper_bytes = archive.read('VSign/' + helper_name)
else:
    raise RuntimeError('Attach đúng một gói seated-v3; không tìm thấy hoặc có nhiều gói.')
expected = next(entry for entry in inventory['files'] if entry['path'] == helper_name)
if len(helper_bytes) != expected['bytes'] or hashlib.sha256(helper_bytes).hexdigest() != expected['sha256']:
    raise RuntimeError('Script setup không khớp checksum của gói.')
with tempfile.TemporaryDirectory(dir=WORK_ROOT) as bootstrap:
    helper = Path(bootstrap) / 'setup.py'
    helper.write_bytes(helper_bytes)
    subprocess.run([sys.executable, str(helper), '--bundle', str(SOURCE), '--output-root', str(ROOT)], check=True)
print('Workspace:', ROOT)


In [ ]:
# Giữ nguyên torch/torchvision CUDA có sẵn trong Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                'av>=12.3.0', 'PyYAML>=6.0.2', 'Pillow>=11.2.1'], check=True)
import torch, torchvision, av
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'av:', av.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Settings -> Accelerator -> GPU trước khi tiếp tục.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
os.chdir(ROOT)
TRAINER = ROOT / 'tmp/train_sales30_seated.py'
BASELINE = ROOT / 'baseline/training/best.pt'
common = [sys.executable, str(TRAINER), '--root', str(ROOT),
          '--dataset-version', 'seated-v3', '--baseline', str(BASELINE)]
subprocess.run(common + ['--prepare-only'], check=True)
subprocess.run(common + ['--smoke-only'], check=True)


## Huấn luyện khi bạn chủ động chạy cell dưới
Cell mặc định tạo run mới. Nếu run đã có dữ liệu, script dừng thay vì tự resume. Khi cần tiếp tục **chính run seated-v3 này**, đặt `RESUME` thành đường dẫn `last.pt`, giữ nguyên epochs/lr và có `best.pt`, `run_identity.json` cùng thư mục. Cặp best/last phải khớp run, best score và history. Không dùng `last.pt` seated-v2.

Kaggle working không bảo đảm tồn tại qua session mới: Save Version với output hoặc tải kết quả về. Khi khôi phục, đặt checkpoint vào thư mục run tương ứng rồi chỉ định RESUME.


In [ ]:
RUN_DIR = WORK_ROOT / 'runs/sales30-seated-v3'
RESUME = None
command = common + ['--output-dir', str(RUN_DIR), '--drive-output-dir', str(RUN_DIR),
                    '--epochs', '6', '--learning-rate', '2e-5',
                    '--batch-size', '2', '--num-workers', '2']
if RESUME is not None:
    command += ['--resume', str(RESUME)]
subprocess.run(command, check=True)


In [ ]:
import shutil
from IPython.display import FileLink, display
summary = json.loads((RUN_DIR / 'summary.json').read_text(encoding='utf-8'))
test = json.loads((RUN_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
print(json.dumps({'summary': summary, 'standing_test': test}, ensure_ascii=False, indent=2))
result_zip = shutil.make_archive(str(WORK_ROOT / 'sales30-seated-v3-results'), 'zip', root_dir=RUN_DIR)
display(FileLink(result_zip))
